# Train & evaluate on Stanford Cars (Colab GPU)

Runtime → Change runtime type → **T4 GPU**. Total time: roughly 1–2 hours for the main runs; the active-learning experiment adds a few more.

Every number this notebook prints is a real measurement — copy them into the README results table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/manojrevenkar12-web/item-identifier.git
%cd item-identifier
!pip -q install -e ".[export,serve,dev]" datasets

In [ ]:
!pytest -q   # sanity check before spending GPU time

## 1. Data
Downloads Stanford Cars (196 models, 49 makes) and the corrupted test copies used for the robustness evaluation.

In [ ]:
!python scripts/prepare_stanford_cars.py --out data/stanford_cars --robustness

## 2. Fine-tune DINOv2 ViT-S/14 and the ResNet-50 baseline

In [ ]:
!itemid train --config configs/stanford_cars.yaml

In [ ]:
!itemid train --config configs/baseline_resnet50.yaml

## 3. Held-out evaluation, robustness, export, latency, gates

In [ ]:
for run in ["runs/cars_dinov2_s", "runs/cars_resnet50"]:
    cfg = "configs/stanford_cars.yaml" if "dinov2" in run else "configs/baseline_resnet50.yaml"
    !itemid evaluate --config {cfg} --checkpoint {run}/model.pt --split test

In [ ]:
import json, glob
corruptions = ["motion_blur", "jpeg_compression", "gaussian_noise", "pixelate", "contrast"]
for c in corruptions:
    !itemid evaluate --config configs/stanford_cars.yaml --checkpoint runs/cars_dinov2_s/model.pt --split test_{c} --out runs/cars_dinov2_s/report_{c}.json > /dev/null
rows = []
for name in ["test"] + corruptions:
    r = json.load(open(f"runs/cars_dinov2_s/report_{name}.json"))
    rows.append((name, r["top1"], r["calibration"]["ece"], r["selective"]["coverage"], r["selective"]["selective_precision"]))
print(f"{'split':20} top1   ECE    coverage  precision@accepted")
for n, a, e, c, p in rows:
    print(f"{n:20} {a:.3f}  {e:.3f}  {c:.3f}     {p if p is None else round(p,3)}")

In [ ]:
!itemid export --checkpoint runs/cars_dinov2_s/model.pt --out runs/cars_dinov2_s/model.onnx
!itemid bench --checkpoint runs/cars_dinov2_s/model.pt --onnx runs/cars_dinov2_s/model.onnx --threads 2 --out runs/cars_dinov2_s/latency.json
!itemid gate --config configs/stanford_cars.yaml --report runs/cars_dinov2_s/report_test.json --latency runs/cars_dinov2_s/latency.json

In [ ]:
!itemid prioritize --report runs/cars_dinov2_s/report_test.json | head -40

## 4. Active-learning experiment (optional, ~2–3 h)

In [ ]:
!itemid al-simulate --config configs/cars_active_learning.yaml --strategies random,margin,margin_diverse --rounds 3 --budget 700 --out runs/al_cars
from IPython.display import Image; Image('runs/al_cars/al_curve.png')

## 5. Save results
Download the zip, then put `model.pt` / `model.onnx` on the machine that serves the API (they are too large for git; use a GitHub release or Hugging Face Hub).

In [ ]:
!zip -qr results.zip runs/*/report_*.json runs/*/report_test.html runs/*/calibration.json runs/*/history.json runs/*/latency.json runs/al_cars/al_results.json runs/al_cars/al_curve.png runs/cars_dinov2_s/model.pt runs/cars_dinov2_s/model.onnx 2>/dev/null
from google.colab import files; files.download('results.zip')